# Colab Runner — DL-MPW Image Captioning

Per-session bootstrap for training `captioning_starter.ipynb` on Google Colab. Run cells top-to-bottom once per fresh runtime.

**Prerequisites (one-time, on the Colab account):**
- Colab Secret `GITLAB_PAT` — GitLab OST personal access token with `read_repository` scope.
- Colab Secret `WANDB_API_KEY` — Weights & Biases API key.
- `flickr8k.zip` uploaded to Drive at `MyDrive/Colab Notebooks/image_captioning_project/data/flickr8k.zip` (must contain an `Images/` folder with the JPGs).

**What this notebook does:**
1. Verifies GPU.
2. Clones the repo from `gitlab.ost.ch` into `/content/image_captioning_project`. The clone already contains `captioning_starter/data/captions.txt` and `captioning_starter/data/splits/`.
3. Installs/upgrades torch, torchvision, nltk, wandb.
4. Copies `flickr8k.zip` from Drive → `/content/`, unzips locally, and moves the JPGs into `captioning_starter/data/images/` inside the cloned repo.
5. Logs into W&B.

Final dataset layout (matches `dataloader.py` expectations):
```
/content/image_captioning_project/captioning_starter/data/
├── captions.txt    # already in the repo
├── images/         # populated from the zip
└── splits/         # already in the repo
```

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

Token is read from Colab Secrets and embedded in the clone URL via `subprocess` so it never appears in cell output or shell history.

In [ ]:
import os, subprocess
from google.colab import userdata

REPO_DIR = "/content/image_captioning_project"
REPO_HOST = "gitlab.ost.ch"
REPO_PATH = "tobias.schuler/image_captioning_project.git"

token = userdata.get("GITLAB_PAT")
if not token:
    raise RuntimeError("Colab Secret 'GITLAB_PAT' is missing or empty.")

clone_url = f"https://oauth2:{token}@{REPO_HOST}/{REPO_PATH}"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", clone_url, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}")

del token, clone_url  # don't keep the token in memory longer than needed

## 3. Install / upgrade dependencies

Colab ships older torch/torchvision than `requirements.txt` pins. The other packages (numpy, pandas, matplotlib, Pillow, tqdm) are already preinstalled on Colab. `wandb` isn't, and `nltk` is needed for BLEU.

In [ ]:
%pip install -q -U "torch>=2.6" "torchvision>=0.21" nltk wandb

## 4. Mount Drive and stage the dataset

Copies `flickr8k.zip` from Drive to local SSD, unzips into a temp folder, and moves the JPGs into `captioning_starter/data/images/` inside the cloned repo. `captions.txt` and `splits/` are already there from the clone, so nothing else needs to be copied.

In [ ]:
import os, shutil, zipfile
from google.colab import drive

drive.mount("/content/drive")

SRC_ZIP     = "/content/drive/MyDrive/Colab Notebooks/image_captioning_project/data/flickr8k.zip"
LOCAL_ZIP   = "/content/flickr8k.zip"
EXTRACT_DIR = "/content/flickr8k_extracted"
IMAGES_DST  = os.path.join(REPO_DIR, "captioning_starter", "data", "images")

os.makedirs(IMAGES_DST, exist_ok=True)
already_populated = any(f.lower().endswith(".jpg") for f in os.listdir(IMAGES_DST))

if already_populated:
    print(f"{IMAGES_DST} already contains JPGs; skipping unzip.")
else:
    if not os.path.exists(SRC_ZIP):
        raise FileNotFoundError(f"Zip not found at {SRC_ZIP}")
    if not os.path.exists(LOCAL_ZIP):
        print(f"Copying {SRC_ZIP} -> {LOCAL_ZIP}")
        shutil.copy(SRC_ZIP, LOCAL_ZIP)
    print(f"Extracting to {EXTRACT_DIR}")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        z.extractall(EXTRACT_DIR)

    src_images = None
    for candidate in ("Images", "images"):
        p = os.path.join(EXTRACT_DIR, candidate)
        if os.path.isdir(p):
            src_images = p
            break
    if src_images is None:
        raise FileNotFoundError(
            f"No Images/ folder inside the zip; extracted contents: {os.listdir(EXTRACT_DIR)}"
        )

    print(f"Moving JPGs: {src_images} -> {IMAGES_DST}")
    for name in os.listdir(src_images):
        shutil.move(os.path.join(src_images, name), os.path.join(IMAGES_DST, name))

    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)

print(f"\n{IMAGES_DST}: {len(os.listdir(IMAGES_DST))} files")

## 5. W&B login

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get("WANDB_API_KEY"))

## 6. Ready — open the training notebook

The training notebook lives at `/content/image_captioning_project/captioning_starter/captioning_starter.ipynb`. Open it from the file browser in the left sidebar.

In the training notebook, override the `data_dir` cell so it points at the staged dataset inside the cloned repo:

```python
data_dir = "/content/image_captioning_project/captioning_starter/data/"
```